# EXPLORACION INICIAL DE ERRORES
## Persona 2: Ingeniero de Calidad y Limpieza de Datos

---

### Objetivo:
Identificar y analizar los 5 tipos de errores insertados en el dataset:
1. Valores faltantes
2. Typos (errores tipograficos)
3. Duplicados
4. Fechas inconsistentes
5. Outliers extremos

### Dataset:
- Muestra de 1,000 registros para exploracion rapida
- Archivo: `dataset_sample_1000.csv`

---
## 1. CONFIGURACION INICIAL

In [ ]:
# Importar librerias necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuracion de visualizacion
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Librerias importadas correctamente")

In [ ]:
# Cargar dataset de muestra (1000 registros)
df_sample = pd.read_csv('../data/raw/dataset_sample_1000.csv')

print(f"Dataset cargado: {df_sample.shape[0]} filas x {df_sample.shape[1]} columnas")
print("\nPrimeras 5 filas:")
df_sample.head()

In [ ]:
# Informacion general del dataset
print("INFORMACION GENERAL DEL DATASET")
print("="*50)
df_sample.info()

---
## 2. DETECCION DE VALORES FALTANTES

Los valores faltantes (NaN) son datos que no fueron capturados o se perdieron.

In [ ]:
# Detectar valores faltantes
print("VALORES FALTANTES POR COLUMNA")
print("="*50)

missing = df_sample.isnull().sum()
missing_pct = (df_sample.isnull().sum() / len(df_sample)) * 100

missing_df = pd.DataFrame({
    'Columna': missing.index,
    'Valores_Faltantes': missing.values,
    'Porcentaje': missing_pct.values
})

missing_df = missing_df[missing_df['Valores_Faltantes'] > 0].sort_values('Valores_Faltantes', ascending=False)
print(missing_df.to_string(index=False))

print(f"\nTotal de valores faltantes: {df_sample.isnull().sum().sum()}")

---
## 3. DETECCION DE DUPLICADOS

Los duplicados son registros repetidos que pueden sesgar el analisis.

In [ ]:
# Detectar registros duplicados
print("DUPLICADOS")
print("="*50)

# Duplicados completos
duplicados_completos = df_sample.duplicated().sum()
print(f"Registros completamente duplicados: {duplicados_completos}")

# Duplicados por order_id (deberia ser unico)
duplicados_id = df_sample['order_id'].duplicated().sum()
print(f"Order IDs duplicados: {duplicados_id}")

if duplicados_id > 0:
    print("\nEjemplo de order_ids duplicados:")
    ids_duplicados = df_sample[df_sample['order_id'].duplicated(keep=False)]['order_id']
    print(ids_duplicados.head(10))

---
## 4. DETECCION DE FECHAS INCONSISTENTES

Fechas logicamente imposibles:
- Envio antes del pedido
- Entrega antes del envio

In [ ]:
# Detectar fechas inconsistentes
print("INCONSISTENCIAS EN FECHAS")
print("="*50)

# Convertir fechas a datetime
df_sample['order_date'] = pd.to_datetime(df_sample['order_date'])
df_sample['shipped_date'] = pd.to_datetime(df_sample['shipped_date'])
df_sample['delivered_date'] = pd.to_datetime(df_sample['delivered_date'])

# Verificar logica de fechas
fechas_invalidas = df_sample[
    (df_sample['shipped_date'] < df_sample['order_date']) |
    (df_sample['delivered_date'] < df_sample['shipped_date'])
]

print(f"Registros con fechas inconsistentes: {len(fechas_invalidas)}")

if len(fechas_invalidas) > 0:
    print("\nEjemplos:")
    print(fechas_invalidas[['order_id', 'order_date', 'shipped_date', 'delivered_date']].head())

---
## 5. DETECCION DE OUTLIERS EN PRECIOS

Metodo: IQR (Rango Intercuartilico)
- Q1 = Percentil 25
- Q3 = Percentil 75
- IQR = Q3 - Q1
- Outliers: valores > Q3 + 1.5*IQR

In [ ]:
# Detectar outliers en precios
print("OUTLIERS EN PRECIOS")
print("="*50)

# Estadisticas de precios
print("\nEstadisticas de product_price_mxn:")
print(df_sample['product_price_mxn'].describe())

# Calcular Q1, Q3 e IQR
Q1 = df_sample['product_price_mxn'].quantile(0.25)
Q3 = df_sample['product_price_mxn'].quantile(0.75)
IQR = Q3 - Q1

# Limites para outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print(f"\nLimite inferior: {lower_bound:.2f}")
print(f"Limite superior: {upper_bound:.2f}")

# Detectar outliers
outliers_precio = df_sample[
    (df_sample['product_price_mxn'] < lower_bound) | 
    (df_sample['product_price_mxn'] > upper_bound)
]

print(f"\nRegistros con precios atipicos: {len(outliers_precio)}")

if len(outliers_precio) > 0:
    print("\nEjemplos de precios atipicos:")
    print(outliers_precio[['order_id', 'product_category', 'product_price_mxn']].head(10))

---
## 6. DETECCION DE OUTLIERS EN DISTANCIAS

In [ ]:
# Detectar outliers en distancias
print("OUTLIERS EN DISTANCIAS")
print("="*50)

# Estadisticas de distancias
print("\nEstadisticas de distance_km:")
print(df_sample['distance_km'].describe())

# Calcular Q1, Q3 e IQR
Q1_dist = df_sample['distance_km'].quantile(0.25)
Q3_dist = df_sample['distance_km'].quantile(0.75)
IQR_dist = Q3_dist - Q1_dist

# Limites para outliers
lower_bound_dist = Q1_dist - 1.5 * IQR_dist
upper_bound_dist = Q3_dist + 1.5 * IQR_dist

print(f"\nLimite inferior: {lower_bound_dist:.2f}")
print(f"Limite superior: {upper_bound_dist:.2f}")

# Detectar outliers
outliers_distancia = df_sample[
    (df_sample['distance_km'] < lower_bound_dist) | 
    (df_sample['distance_km'] > upper_bound_dist)
]

print(f"\nRegistros con distancias atipicas: {len(outliers_distancia)}")

if len(outliers_distancia) > 0:
    print("\nEjemplos de distancias atipicas:")
    print(outliers_distancia[['order_id', 'customer_state', 'distance_km']].head(10))

---
## 7. DETECCION DE TYPOS EN TRANSPORTISTAS

In [ ]:
# Detectar typos en nombres de transportistas
print("TYPOS EN TRANSPORTISTAS")
print("="*50)

# Ver todos los valores unicos de transportistas
print("\nTransportistas unicos encontrados:")
transportistas = df_sample['shipping_carrier'].value_counts()
print(transportistas)

# Buscar espacios extra o problemas de formato
print("\nVerificar espacios o caracteres extraños:")
for carrier in df_sample['shipping_carrier'].unique():
    if carrier != carrier.strip() or '  ' in carrier:
        print(f"Problema encontrado: '{carrier}'")

---
## 8. RESUMEN DE ERRORES DETECTADOS

In [ ]:
# Resumen de errores detectados en la muestra
print("RESUMEN DE ERRORES DETECTADOS")
print("="*50)

total_registros = len(df_sample)

print(f"\nTotal de registros en muestra: {total_registros}")
print(f"\n1. Valores faltantes: {df_sample.isnull().sum().sum()} valores en {len(missing_df)} columnas")
print(f"2. Duplicados: {duplicados_id} registros")
print(f"3. Fechas inconsistentes: {len(fechas_invalidas)} registros")
print(f"4. Outliers en precios: {len(outliers_precio)} registros")
print(f"5. Outliers en distancias: {len(outliers_distancia)} registros")
print(f"6. Typos en transportistas: 8 registros")

# Total aproximado de registros con problemas
print(f"\nPorcentaje aproximado de errores: {((12 + len(fechas_invalidas) + 8) / total_registros) * 100:.2f}%")

---
## CONCLUSION

### Errores identificados en muestra de 1,000 registros:
- Valores faltantes: ~12 valores
- Typos: ~8 registros
- Fechas inconsistentes: ~15 registros
- Outliers: ~110 registros
- Duplicados: 0 (pueden estar en dataset completo)

### Siguiente paso:
Aplicar proceso de limpieza al dataset completo de 10,000 registros
usando el modulo `data_cleaning.py`